# Flight Delay Intelligence Platform (FDIP)
## HSLU Data Warehouse and Data Lake — HS 2025
**Author:** Simao Garcia  
**AWS Account:** 604415812723 | **Region:** 

---

This notebook contains all Python code used in the FDIP project:
1. **Lambda Function 1** — AviationStack API ingestion
2. **Lambda Function 2** — NOAA METAR API ingestion  
3. **AWS Glue ETL Job 1** — bronze-to-silver transformation
4. **AWS Glue ETL Job 2** — silver-to-gold transformation

> **Note:** Each section runs as a separate AWS service in production.
> This notebook is for documentation and review purposes.


## Architecture Overview

```
Data Sources → Lambda (00:00) → S3 Bronze → Glue B→S (00:30) → S3 Silver → Glue S→G (01:00) → S3 Gold → Athena → Tableau
```

| Layer | Bucket | Format |
|---|---|---|
| Bronze | s3://bronze-604415812723/ | JSON + CSV |
| Silver | s3://silver-604415812723/ | Parquet/Snappy |
| Gold   | s3://gold-604415812723/   | Parquet/Snappy |


## Dependencies and Imports

In [ ]:
# Standard library imports used across all sections
import json
import boto3
import urllib.request
import urllib.error
import datetime
import os

# PySpark imports (available in AWS Glue runtime)
# Uncomment below if running locally with PySpark installed
# from awsglue.context import GlueContext
# from pyspark.context import SparkContext
# from pyspark.sql import functions as F

print("Imports loaded successfully")
print("Note: AWS Glue imports require the Glue runtime environment")

## Section 1 — Lambda Function: aviationstack-ingest

**Trigger:** Amazon EventBridge — `cron(0 22 * * ? *)` — 00:00 Zurich  
**Runtime:** Python 3.11 | **Timeout:** 60 seconds  
**Output:** `s3://bronze-604415812723/aviationstack/yyyy/mm/dd/snapshot.json`

Polls the AviationStack REST API at `/v1/flights` and extracts:
- flight_status, dep_delay, arr_delay
- airline_code, flight_iata, aircraft_type  
- origin, destination, scheduled_dep


In [ ]:
def lambda_handler_aviationstack(event, context):
    """
    Polls the AviationStack REST API and writes a date-partitioned
    JSON snapshot to the S3 bronze zone.
    Returns the number of flights ingested.
    """

    # Initialise S3 client
    S3 = boto3.client('s3')

    # Read configuration from environment variables
    BUCKET  = os.environ['BRONZE_BUCKET']       # S3 bucket name
    API_KEY = os.environ['AVIATIONSTACK_KEY']   # AviationStack API key

    try:
        # Build the API URL
        # flight_status=active: only airborne flights
        # limit=100: free tier maximum records per call
        url = (
            f'http://api.aviationstack.com/v1/flights'
            f'?access_key={API_KEY}'
            f'&flight_status=active'
            f'&limit=100'
        )

        # Make the HTTP request with a User-Agent header
        req = urllib.request.Request(
            url,
            headers={'User-Agent': 'Mozilla/5.0'}
        )

        # Execute the request with a 25-second timeout
        with urllib.request.urlopen(req, timeout=25) as r:
            data = json.loads(r.read())

        # Build the date-partitioned S3 key
        # One file per day — re-running overwrites only that day's file
        now = datetime.datetime.utcnow()
        key = (
            f'aviationstack/'
            f'{now.year}/{now.month:02}/{now.day:02}'
            f'/snapshot.json'
        )

        # Write raw data plus ingestion timestamp to bronze zone
        S3.put_object(
            Bucket=BUCKET,
            Key=key,
            Body=json.dumps({
                'flights': data,
                '_ingested_at': now.isoformat()
            })
        )

        return {
            'statusCode': 200,
            'flights': len(data.get('data', []))
        }

    except urllib.error.URLError as e:
        # Log error to CloudWatch and re-raise for EventBridge failure detection
        print(f'API call failed: {e}')
        raise


print("aviationstack-ingest function defined")
print("Deploy this function to AWS Lambda with runtime Python 3.11")

## Section 2 — Lambda Function: metar-ingest

**Trigger:** Amazon EventBridge — `cron(0 22 * * ? *)` — 00:00 Zurich  
**Runtime:** Python 3.11 | **Timeout:** 60 seconds  
**Output:** `s3://bronze-604415812723/metar/yyyy/mm/dd/snapshot.json`  
**Auth:** None required — NOAA METAR API is fully public

Polls the NOAA Aviation Weather Center API and extracts for 20 airports:
- icaoId, visibility_sm, wind_kt
- temp_c, ceiling_ft, obsTime
- flight_category (VFR / MVFR / IFR / LIFR)


In [ ]:
# 20 major US airports by ICAO code
AIRPORTS = [
    'KORD', 'KJFK', 'KLAX', 'KSFO', 'KATL',
    'KDEN', 'KDFW', 'KMIA', 'KBOS', 'KSEA',
    'KPHX', 'KEWR', 'KLGA', 'KIAH', 'KFLL',
    'KMCO', 'KMDW', 'KDTW', 'KMSP', 'KBWI'
]

def lambda_handler_metar(event, context):
    """
    Polls the NOAA METAR public API for 20 major US airports
    and writes a date-partitioned JSON snapshot to the S3 bronze zone.
    """

    S3     = boto3.client('s3')
    BUCKET = os.environ['BRONZE_BUCKET']

    try:
        ids = ','.join(AIRPORTS)

        # NOAA public endpoint — no API key required
        url = (
            f'https://aviationweather.gov/api/data/metar'
            f'?ids={ids}&format=json'
        )

        req = urllib.request.Request(
            url,
            headers={'User-Agent': 'Mozilla/5.0'}
        )

        with urllib.request.urlopen(req, timeout=25) as r:
            data = json.loads(r.read())

        now = datetime.datetime.utcnow()
        key = (
            f'metar/'
            f'{now.year}/{now.month:02}/{now.day:02}'
            f'/snapshot.json'
        )

        S3.put_object(
            Bucket=BUCKET,
            Key=key,
            Body=json.dumps({
                'records': data,
                '_ingested_at': now.isoformat()
            })
        )

        return {
            'statusCode': 200,
            'airports': len(data)
        }

    except urllib.error.URLError as e:
        print(f'API call failed: {e}')
        raise


print("metar-ingest function defined")
print("Deploy this function to AWS Lambda with runtime Python 3.11")

## Section 3 — AWS Glue ETL: bronze-to-silver

**Schedule:** `cron(30 22 * * ? *)` — 00:30 Zurich  
**Worker:** G.1X | **Workers:** 4 | **Glue version:** 5.1  
**Input:** `s3://bronze-604415812723/`  
**Output:** `s3://silver-604415812723/`

Processes three sources in independent fault-isolated blocks:
| Source | Mode | Output |
|---|---|---|
| AviationStack JSON | append | aviationstack_flights/ |
| METAR JSON | append | metar/ |
| BTS CSV | overwrite | bts_flights/ (75 MB, 43× compression) |


### 3.1 AviationStack → Silver

In [ ]:
# Initialize Spark context (runs automatically in AWS Glue)
# from awsglue.context import GlueContext
# from pyspark.context import SparkContext
# from pyspark.sql import functions as F
# sc = SparkContext()
# gc = GlueContext(sc)
# spark = gc.spark_session

BRONZE = 's3://bronze-604415812723/'
SILVER = 's3://silver-604415812723/'

# --- AviationStack Silver ---
try:
    aviation = spark.read.json(BRONZE + 'aviationstack/*/*/*/snapshot.json')

    # explode() flattens the nested flights.data array
    # producing one row per flight record
    aviation_flat = (aviation
        .select(F.explode('flights.data').alias('f'), '_ingested_at')
        .select(
            F.col('f.flight_date').alias('flight_date'),
            F.col('f.flight_status').alias('flight_status'),
            F.col('f.departure.iata').alias('origin'),
            F.col('f.departure.airport').alias('origin_airport'),
            F.col('f.departure.delay').cast('double').alias('dep_delay'),
            F.col('f.departure.scheduled').alias('scheduled_dep'),
            F.col('f.arrival.iata').alias('dest'),
            F.col('f.arrival.delay').cast('double').alias('arr_delay'),
            F.col('f.airline.iata').alias('carrier'),
            F.col('f.flight.iata').alias('flight_iata'),
            F.col('f.aircraft.iata').alias('aircraft_type'),
            F.col('_ingested_at')
        )
        .filter(F.col('origin').isNotNull())
        .dropDuplicates(['flight_date', 'flight_iata', 'origin', 'dest'])
    )

    # Append mode: accumulate daily snapshots over time
    aviation_flat.write.mode('append').parquet(SILVER + 'aviationstack_flights/')
    print(f'AviationStack silver done: {aviation_flat.count()} records')

except Exception as e:
    print(f'AviationStack error: {e}')

### 3.2 METAR → Silver

In [ ]:
# --- METAR Silver ---
try:
    metar = spark.read.json(BRONZE + 'metar/*/*/*/snapshot.json')

    metar_flat = (metar
        .select(F.explode('records').alias('r'), '_ingested_at')
        .select(
            F.col('r.icaoId').alias('airport_icao'),
            F.col('r.obsTime').alias('obs_time'),
            F.col('r.temp').cast('double').alias('temp_c'),
            F.col('r.wspd').cast('double').alias('wind_kt'),
            F.col('r.visib').cast('double').alias('visibility_sm'),
            F.col('r.fltCat').alias('flight_category'),
            F.col('r.rawOb').alias('raw_text'),
            F.col('_ingested_at')
        )
        .filter(F.col('airport_icao').isNotNull())
    )

    metar_flat.write.mode('append').parquet(SILVER + 'metar/')
    print(f'METAR silver done: {metar_flat.count()} records')

except Exception as e:
    print(f'METAR error: {e}')

### 3.3 BTS CSV → Silver

In [ ]:
# --- BTS Silver ---
try:
    # Wildcard reads all 13 monthly files automatically
    bts = spark.read.option('header', True).csv(BRONZE + 'bts/*.csv')

    # Log column names for schema drift detection
    print('BTS columns:', bts.columns)

    bts_clean = (bts
        .select(
            'FlightDate', 'Reporting_Airline', 'Tail_Number',
            'Origin', 'Dest', 'DepTime', 'DepDelay',
            'ArrTime', 'ArrDelay', 'Cancelled', 'CancellationCode',
            'CarrierDelay', 'WeatherDelay', 'NASDelay',
            'SecurityDelay', 'LateAircraftDelay'
        )
        # Rename to internal pipeline standard
        .withColumnRenamed('FlightDate',         'FL_DATE')
        .withColumnRenamed('Reporting_Airline',  'OP_CARRIER')
        .withColumnRenamed('Tail_Number',        'TAIL_NUM')
        .withColumnRenamed('DepTime',            'DEP_TIME')
        .withColumnRenamed('DepDelay',           'DEP_DELAY')
        .withColumnRenamed('ArrTime',            'ARR_TIME')
        .withColumnRenamed('ArrDelay',           'ARR_DELAY')
        .withColumnRenamed('CancellationCode',   'CANCELLATION_CODE')
        .withColumnRenamed('CarrierDelay',       'CARRIER_DELAY')
        .withColumnRenamed('WeatherDelay',       'WEATHER_DELAY')
        .withColumnRenamed('NASDelay',           'NAS_DELAY')
        .withColumnRenamed('SecurityDelay',      'SECURITY_DELAY')
        .withColumnRenamed('LateAircraftDelay',  'LATE_AIRCRAFT_DELAY')
        # Cast delay columns from string to double
        # Invalid values become null — not job failures
        .withColumn('DEP_DELAY',           F.col('DEP_DELAY').cast('double'))
        .withColumn('ARR_DELAY',           F.col('ARR_DELAY').cast('double'))
        .withColumn('CARRIER_DELAY',       F.col('CARRIER_DELAY').cast('double'))
        .withColumn('WEATHER_DELAY',       F.col('WEATHER_DELAY').cast('double'))
        .withColumn('NAS_DELAY',           F.col('NAS_DELAY').cast('double'))
        .withColumn('SECURITY_DELAY',      F.col('SECURITY_DELAY').cast('double'))
        .withColumn('LATE_AIRCRAFT_DELAY', F.col('LATE_AIRCRAFT_DELAY').cast('double'))
        # FAA standard: > 15 minutes = delayed
        .withColumn('is_delayed', F.col('DEP_DELAY') > 15)
        .withColumn('is_cancelled', F.col('Cancelled').cast('boolean'))
        .filter(F.col('Origin').isNotNull())
        .dropDuplicates(['FL_DATE', 'OP_CARRIER', 'Origin', 'Dest', 'DEP_TIME'])
    )

    # Overwrite: full reload required due to BTS retroactive corrections
    bts_clean.write.mode('overwrite').parquet(SILVER + 'bts_flights/')
    print(f'BTS silver done: {bts_clean.count()} records')
    # Result: 3.2 GB CSV -> 75 MB Parquet (43x compression)

except Exception as e:
    print(f'BTS error: {e}')

print('Bronze to Silver ETL complete!')

## Section 4 — AWS Glue ETL: silver-to-gold

**Schedule:** `cron(0 23 * * ? *)` — 01:00 Zurich  
**Worker:** G.1X | **Workers:** 4 | **Glue version:** 5.1  
**Input:** `s3://silver-604415812723/`  
**Output:** `s3://gold-604415812723/`

| Gold Table | BI# | Key Aggregation | Rows |
|---|---|---|---|
| airport_hourly_delay | BI1 | GROUP BY airport + hour + month | ~40K |
| route_delay_summary | BI2, BI6 | GROUP BY carrier + origin + dest | ~28K |
| carrier_performance | BI4, BI7 | GROUP BY carrier | 14 |
| delay_cause_breakdown | BI5 | Filter delayed + GROUP BY airport + carrier | ~12K |
| flight_weather_joined | BI3 | LEFT JOIN bts + metar (ICAO fix) | 7.4M |
| aviationstack_summary | Live | GROUP BY carrier + origin + dest | Daily |


### 4.1 Airport Hourly Delay (BI1)

In [ ]:
GOLD = 's3://gold-604415812723/'

# Read silver tables
bts   = spark.read.parquet(SILVER + 'bts_flights/')
metar = spark.read.parquet(SILVER + 'metar/')

# Gold 1: Airport Hourly Delay
try:
    airport_hourly = (bts
        # Extract hour from DEP_TIME (hhmm format)
        .withColumn('dep_hour',
            F.expr("CAST(SUBSTR(LPAD(DEP_TIME, 4, '0'), 1, 2) AS INT)"))
        .withColumn('month',
            F.month(F.to_date('FL_DATE', 'yyyy-MM-dd')))
        .groupBy('Origin', 'dep_hour', 'month')
        .agg(
            F.avg('DEP_DELAY').alias('avg_delay'),
            F.count('*').alias('total_flights'),
            (F.sum(F.col('is_delayed').cast('int')) /
             F.count('*')).alias('delay_rate')
        )
    )
    airport_hourly.write.mode('overwrite').parquet(GOLD + 'airport_hourly_delay/')
    print(f'Gold airport hourly: {airport_hourly.count()} records')
except Exception as e: print(f'Airport hourly error: {e}')

### 4.2 Route Delay Summary (BI2, BI6)

In [ ]:
# Gold 2: Route Delay Summary
try:
    route_summary = (bts
        .groupBy('OP_CARRIER', 'Origin', 'Dest')
        .agg(
            F.avg('DEP_DELAY').alias('avg_dep_delay'),
            F.count('*').alias('total_flights'),
            (F.sum(F.col('is_delayed').cast('int')) /
             F.count('*')).alias('delay_rate'),
            F.avg('CARRIER_DELAY').alias('avg_carrier_delay'),
            F.avg('WEATHER_DELAY').alias('avg_weather_delay'),
            F.avg('NAS_DELAY').alias('avg_nas_delay'),
            F.avg('LATE_AIRCRAFT_DELAY').alias('avg_late_aircraft_delay')
        )
        # Minimum 10 flights for statistical validity
        .filter(F.col('total_flights') >= 10)
    )
    route_summary.write.mode('overwrite').parquet(GOLD + 'route_delay_summary/')
    print(f'Gold route summary: {route_summary.count()} routes')
except Exception as e: print(f'Route summary error: {e}')

### 4.3 Carrier Performance (BI4, BI7)

In [ ]:
# Gold 3: Carrier Performance
try:
    carrier_perf = (bts
        .groupBy('OP_CARRIER')
        .agg(
            F.avg('DEP_DELAY').alias('avg_dep_delay'),
            F.avg('ARR_DELAY').alias('avg_arr_delay'),
            F.count('*').alias('total_flights'),
            (F.sum(F.col('is_delayed').cast('int')) /
             F.count('*')).alias('delay_rate'),
            F.sum(F.col('is_cancelled').cast('int')).alias('total_cancelled')
        )
        .orderBy('delay_rate')
    )
    carrier_perf.write.mode('overwrite').parquet(GOLD + 'carrier_performance/')
    print(f'Gold carrier performance: {carrier_perf.count()} carriers')
except Exception as e: print(f'Carrier perf error: {e}')

### 4.4 Delay Cause Breakdown (BI5)

In [ ]:
# Gold 4: Delay Cause Breakdown
try:
    delay_causes = (bts
        # Filter to delayed flights only
        .filter(F.col('is_delayed') == True)
        .groupBy('Origin', 'OP_CARRIER')
        .agg(
            F.avg('CARRIER_DELAY').alias('avg_carrier_delay'),
            F.avg('WEATHER_DELAY').alias('avg_weather_delay'),
            F.avg('NAS_DELAY').alias('avg_nas_delay'),
            F.avg('LATE_AIRCRAFT_DELAY').alias('avg_late_aircraft_delay'),
            F.count('*').alias('total_delayed_flights')
        )
        # Minimum 5 delayed flights for statistical validity
        .filter(F.col('total_delayed_flights') >= 5)
    )
    delay_causes.write.mode('overwrite').parquet(GOLD + 'delay_cause_breakdown/')
    print(f'Gold delay causes: {delay_causes.count()} records')
except Exception as e: print(f'Delay causes error: {e}')

### 4.5 Flight Weather Joined (BI3) — CORRECTED JOIN

**Bug fix:** BTS uses IATA codes (e.g. `ORD`) but METAR uses ICAO codes (e.g. `KORD`).  
**Solution:** Add `K` prefix to BTS Origin before joining so `ORD` → `KORD` matches METAR.


In [ ]:
# Gold 5: Flight Weather Joined — CORRECTED JOIN
try:
    flight_weather = (bts
        # Add K prefix to match METAR ICAO codes
        # BTS: ORD → KORD (ICAO) → matches METAR airport_icao
        .withColumn('origin_icao',
            F.concat(F.lit('K'), F.col('Origin')))
        # Left join: ALL 7.4M BTS flights retained
        # Flights without METAR match get null weather fields
        .join(metar,
              F.col('origin_icao') == metar['airport_icao'],
              'left')
        .select(
            bts['FL_DATE'],
            bts['OP_CARRIER'],
            bts['Origin'],
            bts['Dest'],
            bts['DEP_DELAY'],
            bts['is_delayed'],
            metar['visibility_sm'],
            metar['flight_category'],   # VFR / MVFR / IFR / LIFR
            metar['temp_c'],
            metar['wind_kt']
        )
        .drop('origin_icao')            # Remove temporary join key
    )
    flight_weather.write.mode('overwrite').parquet(GOLD + 'flight_weather_joined/')
    print(f'Gold flight weather: {flight_weather.count()} records')
except Exception as e: print(f'Flight weather error: {e}')

### 4.6 AviationStack Summary (Live)

In [ ]:
# Gold 6: AviationStack Summary
try:
    aviation = spark.read.parquet(SILVER + 'aviationstack_flights/')
    aviation_summary = (aviation
        .groupBy('carrier', 'origin', 'dest')
        .agg(
            F.count('*').alias('total_flights'),
            F.avg('dep_delay').alias('avg_dep_delay'),
            F.avg('arr_delay').alias('avg_arr_delay'),
            F.sum(F.when(F.col('dep_delay') > 15, 1)
                   .otherwise(0)).alias('delayed_flights'),
            F.first('aircraft_type').alias('aircraft_type')
        )
        .filter(F.col('total_flights') >= 2)
    )
    aviation_summary.write.mode('overwrite').parquet(GOLD + 'aviationstack_summary/')
    print(f'Gold AviationStack: {aviation_summary.count()} routes')
except Exception as e: print(f'AviationStack error: {e}')

print('Silver to Gold ETL complete!')

---
## Deployment Notes

| Section | Deploy as | Schedule |
|---|---|---|
| Section 1 | AWS Lambda (aviationstack-ingest) | EventBridge cron(0 22 * * ? *) |
| Section 2 | AWS Lambda (metar-ingest) | EventBridge cron(0 22 * * ? *) |
| Section 3 | AWS Glue Job (bronze-to-silver) | Glue Scheduler cron(30 22 * * ? *) |
| Section 4 | AWS Glue Job (silver-to-gold) | Glue Scheduler cron(0 23 * * ? *) |

  
**AWS Account:** 604415812723 | **Region:** us-east-1
